# Lecture 18: Practice Exercises
### NLP Course 2027 — Week 09

---

These exercises accompany **Lecture 18: Productionizing NLP Systems**.
Complete each exercise in the provided code cell.

In [ ]:
import time
import json
import torch
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from functools import lru_cache

---

## Exercise 18.1

Build a complete FastAPI service with `/sentiment`, `/ner`, and `/summarize` endpoints. Add request logging and error handling.

In [ ]:
# Exercise 18.1
# Write the FastAPI app code. Save to a file and run with uvicorn.

fastapi_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline
import time
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="NLP API v2", version="2.0.0")

class TextRequest(BaseModel):
    text: str

@app.on_event("startup")
async def load_models():
    # YOUR CODE: load sentiment, NER, and summarization models
    pass

@app.post("/sentiment")
async def sentiment(req: TextRequest):
    # YOUR CODE: validate input, run model, log request, return result
    pass

@app.post("/ner")
async def ner(req: TextRequest):
    # YOUR CODE
    pass

@app.post("/summarize")
async def summarize(req: TextRequest):
    # YOUR CODE: add min_length check (e.g., at least 50 words)
    pass

@app.get("/health")
async def health():
    return {"status": "healthy"}

# To run: uvicorn solution:app --reload
'''

with open('/tmp/nlp_api.py', 'w') as f:
    f.write(fastapi_code)
print('FastAPI app written to /tmp/nlp_api.py')
print('Run with: uvicorn /tmp/nlp_api:app --reload')


---

## Exercise 18.2

Benchmark your model: measure requests/second, p50/p95/p99 latency. Use `asyncio` for load testing.

In [ ]:
# Exercise 18.2
import time
import numpy as np
from transformers import pipeline

sentiment = pipeline('text-classification',
                     model='distilbert-base-uncased-finetuned-sst-2-english')

def benchmark_model(model_fn, texts, n_runs=3):
    """
    Benchmark a model function.
    Returns: avg latency, p50, p95, p99, throughput (docs/sec)
    """
    # YOUR CODE: measure latency for each text, compute percentiles
    latencies = []
    for _ in range(n_runs):
        for text in texts:
            t0 = time.time()
            model_fn(text)
            latencies.append((time.time() - t0) * 1000)
    latencies = sorted(latencies)
    return {
        'avg_ms': np.mean(latencies),
        'p50_ms': latencies[int(len(latencies) * 0.50)],
        'p95_ms': latencies[int(len(latencies) * 0.95)],
        'p99_ms': latencies[int(len(latencies) * 0.99)],
        'throughput': 1000 / np.mean(latencies),
    }

texts = [
    'This product is excellent!',
    'Terrible experience, would not recommend.',
    'Average quality, nothing special.',
] * 20

results = benchmark_model(lambda t: sentiment(t), texts)
print('Benchmark Results:')
for k, v in results.items():
    print(f'  {k}: {v:.2f}')


---

## Exercise 18.3

Implement prediction caching with Redis (`pip install redis`). How does cache hit rate change as you increase traffic on repeated queries?

In [ ]:
# Exercise 18.3
# Option A: Use Redis (if available)
# Option B: Simulate caching with a dict and measure cache hit rate

class PredictionCache:
    """LRU-based prediction cache (simulates Redis behavior)."""

    def __init__(self, model_fn, max_size=500):
        self.model_fn = model_fn
        self.max_size = max_size
        self._cache = {}
        self.hits = 0
        self.misses = 0

    def predict(self, text):
        if text in self._cache:
            self.hits += 1
            return self._cache[text]
        result = self.model_fn(text)
        if len(self._cache) >= self.max_size:
            # Evict oldest entry
            self._cache.pop(next(iter(self._cache)))
        self._cache[text] = result
        self.misses += 1
        return result

    def stats(self):
        total = self.hits + self.misses
        return {
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': self.hits / total if total > 0 else 0,
        }

# YOUR CODE: simulate traffic with varying repetition rates
# e.g., 10%, 30%, 50%, 70% of requests are repeated queries
# Plot cache hit rate vs repetition rate


---

## Exercise 18.4

Set up MLflow to track experiments for Lecture 13's fine-tuning task. Log hyperparameters, metrics, and save the best model.

In [ ]:
# Exercise 18.4
# !pip install mlflow

mlflow_code = """
import mlflow
import mlflow.pytorch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
import numpy as np

mlflow.set_experiment('nlp-fine-tuning-lecture13')

hyperparams = [
    {'lr': 2e-5, 'epochs': 3, 'batch_size': 32},
    {'lr': 5e-5, 'epochs': 3, 'batch_size': 16},
    {'lr': 2e-5, 'epochs': 5, 'batch_size': 32},
]

for hp in hyperparams:
    with mlflow.start_run(run_name=f"lr={hp['lr']}_ep={hp['epochs']}"):
        mlflow.log_params(hp)
        # YOUR CODE: train model with these hyperparams
        # Log: mlflow.log_metric('val_accuracy', val_acc)
        # Save: mlflow.pytorch.log_model(model, 'model')
"""

print('MLflow experiment tracking code:')
print(mlflow_code)
# YOUR CODE: implement and run the experiment tracking


---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**